### Chatbot with message History


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")


In [2]:
from langchain_groq import ChatGroq
model = ChatGroq(model="openai/gpt-oss-120b",groq_api_key=groq_api_key)
model

d:\GenAi\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000025560807C80>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000255609D49E0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi My name is Kheman Saru and I am currently studying engineering")])

AIMessage(content='Hello Kheman! 👋 It’s great to meet you. How’s engineering life treating you so far? Which branch are you studying—mechanical, electrical, civil, computer, or something else? If you have any questions, need resources, or just want to chat about your projects, I’m here to help!', additional_kwargs={'reasoning_content': 'We need to respond. The user says "Hi My name is Kheman Saru and I am currently studying engineering". Likely they want a greeting, maybe ask about their field, offer help. We should respond politely, ask about their engineering discipline, how we can assist. No disallowed content. So just a friendly response.'}, response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens': 86, 'total_tokens': 230, 'completion_time': 0.312660298, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.003256504, 'prompt_tokens_details': None, 'queue_time': 0.044658195, 'total_time': 0.315916802}, 'model_name': 'openai/gpt-oss-120b', 'syst

In [4]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]
with_message_history = RunnableWithMessageHistory(model,get_session_history)

In [5]:
config = {"configurable":{"session_id":"chat_1"}}

In [6]:
response = with_message_history.invoke(
            [HumanMessage(content="Hi My name is Kheman Saru and I am currently studying engineering")],
            config=config
        )   

In [7]:
response.content

'Hello Kheman! 👋 It’s great to meet you. How’s engineering school going so far? What branch are you studying, and is there anything specific you’d like to chat about or get help with?'

In [8]:
with_message_history.invoke(
            [HumanMessage(content="What is my name?")],
            config=config
        )   

AIMessage(content='Your name is Kheman\u202fSaru.', additional_kwargs={'reasoning_content': 'The user asks "What is my name?" The assistant should answer based on conversation. The user introduced themselves as Kheman Saru. So answer: Kheman Saru.'}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 145, 'total_tokens': 202, 'completion_time': 0.120163883, 'completion_tokens_details': {'reasoning_tokens': 38}, 'prompt_time': 0.008835439, 'prompt_tokens_details': None, 'queue_time': 0.052344761, 'total_time': 0.128999322}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a09bde29de', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cc8ec-7e20-7fd0-b273-92582d3e7ca8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 145, 'output_tokens': 57, 'total_tokens': 202, 'output_token_details': {'reasoning': 38}})

In [9]:
config1 = {"configurable":{"session_id":"chat_2"}}
response = with_message_history.invoke(
            [HumanMessage(content="What is my name?")],
            config=config1
        )   
response.content

'I don’t have any information about your name. If you’d like to share it, feel free to let me know!'

In [10]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        ("system""You are an ai assistant.Answer all the query like a teacher with nice tone"),
        MessagesPlaceholder(variable_name="messages")
    ]
)
chain = prompt|model


In [11]:
chain.invoke({"messages":[HumanMessage(content="Hi my name is Kheman")]})


AIMessage(content="Hello, Kheman! 👋 It's wonderful to meet you. How can I assist you today? Feel free to ask any questions—whether they're about a subject you're studying, a curiosity you have, or anything else you’d like to explore. I'm here to help!", additional_kwargs={'reasoning_content': 'We need to respond as a teacher with a nice tone, greeting the user, acknowledging their name. Probably ask how can help.'}, response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 99, 'total_tokens': 190, 'completion_time': 0.19196021, 'completion_tokens_details': {'reasoning_tokens': 27}, 'prompt_time': 0.00368298, 'prompt_tokens_details': None, 'queue_time': 0.04482187, 'total_time': 0.19564319}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_626f3fc5e0', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cc8ec-80cf-75e1-9862-652a10ee7607-0', tool_calls=[], invalid_tool_calls=[], usage_metadata=

In [12]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [13]:
config = {"configurable":{"session_id":"chat_4"}}
response=with_message_history.invoke(
    [HumanMessage(content="what is my name?")],
    config=config
)
response

AIMessage(content='Hello! I’m glad you asked. Since we haven’t introduced ourselves yet, I don’t know your name at the moment. Could you tell me what you’d like to be called? Once I know it, I’ll be happy to address you properly in our conversation. 😊', additional_kwargs={'reasoning_content': 'The user says "systemYou are an ai assistant.Answer all the query like a teacher with nice tone". Then asks "what is my name?" There\'s no prior context with their name. As an AI, I must respond that I don\'t know their name unless they tell me. Should be nice teacher tone. Also note system instruction: "Answer all the query like a teacher with nice tone". So respond politely, maybe ask them to tell me their name.'}, response_metadata={'token_usage': {'completion_tokens': 158, 'prompt_tokens': 97, 'total_tokens': 255, 'completion_time': 0.330229319, 'completion_tokens_details': {'reasoning_tokens': 93}, 'prompt_time': 0.003934295, 'prompt_tokens_details': None, 'queue_time': 0.045027295, 'total_t

In [14]:
from langchain_core.messages import trim_messages,SystemMessage,AIMessage
trimmer = trim_messages(
    max_tokens=100,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you are a good assistant"),
    HumanMessage(content="hi I am Kheman"),
    AIMessage(content="hi"),
    HumanMessage(content="I like vanilla icecream"),
    AIMessage(content="nice"),
    HumanMessage(content="what's 2+2"),
    AIMessage(content="4"),
    HumanMessage(content="Thanks"),
    AIMessage(content="No problem"),
    HumanMessage(content="having Fun"),
    AIMessage(content="yes")
]
trimmer.invoke(messages)

d:\GenAi\venv\Lib\site-packages\langchain_core\language_models\base.py:336: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


[SystemMessage(content='you are a good assistant', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='hi I am Kheman', additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I like vanilla icecream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content="what's 2+2", additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='No problem', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having Fun', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes', additional_kwargs={

In [15]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
chain = (RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
         |prompt
         |model
         
         )
response = chain.invoke(
    {
                "messages":messages + [HumanMessage(content="Which icecream do i like?")],
                "language":"English"
    }
            )
response.content

'You mentioned earlier that you enjoy **vanilla ice cream**—so that’s the flavor you like! If you ever want to explore new varieties or learn about the history of vanilla, just let me know. 😊'

In [16]:
from langchain_core.documents import Document
documents=[Document(
    page_content="Dogs are grat companion,known for their loyalty and friendship",
    metadata={"source":"mammals-pets-docs"}
    ),
    Document(
        page_content="Cats are independent pets and enjoys their personal space",
        metadata={"source":"mammals-pets-docs"}
    ),
    Document(
        page_content="Goldfish are the popualar pets for beginners,requiring relatively simple care",
        metadata={"source":"fish-pets-docs"}
    ),
    Document(
        page_content="Parrot are inteligent bird capable of mimicking Human Beings",
        metadata={"source":"bird-pets-docs"}
    )


]

In [17]:
documents

[Document(metadata={'source': 'mammals-pets-docs'}, page_content='Dogs are grat companion,known for their loyalty and friendship'),
 Document(metadata={'source': 'mammals-pets-docs'}, page_content='Cats are independent pets and enjoys their personal space'),
 Document(metadata={'source': 'fish-pets-docs'}, page_content='Goldfish are the popualar pets for beginners,requiring relatively simple care'),
 Document(metadata={'source': 'bird-pets-docs'}, page_content='Parrot are inteligent bird capable of mimicking Human Beings')]